# Read new data using Auto Loader
Script Purpose:
* Read new, unmodifed data 
* Add information about ingestion timestamp
* Write data to bronze layer

In [0]:
from pyspark.sql import functions as F

With the use of Databricks Auto Loader:
* Read csv files
* Keep Auto Loader's ingested data schema
* Keep headers from csv files
* Load data from the source (folder with generated data) 

In [0]:
files = {
    "customers": ["gen_customers", "_customer_data"],
    "hubs":["gen_hubs", "_hubs_data"],
    "shipping":["gen_shipments", "_shipping_data"]
}

In [0]:
dfs = []
for key, val in files.items():
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"/Volumes/deliveries/default/checkpoint/{val[0]}/schema")
        .option("header", "true")
        .load(f"/Volumes/deliveries/default/generated{val[1]}/")
    )
    dfs.append(df)

Add a timestamp of ingestion time

In [0]:
for df in dfs:
    df = df.withColumn("ingested_at", F.current_timestamp())


Write stream into the table:
* Delta table to keep historic data
* Append mode to add new data to table
* Auto Loader's checkpoint location
* Trigger to stop process after code finishes the run
* Write DataFrame into bronze layer

In [0]:

for index, (key, val) in enumerate(files.items()):
    df = dfs[index]
    (df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"/Volumes/deliveries/default/checkpoint/{val[0]}")
    .trigger(availableNow=True)
    .toTable(f"deliveries.bronze.{val[0]}"))

See the outcome data table

In [0]:
for key, val in files.items():
    df_bronze = spark.read.table(f"deliveries.bronze.{val[0]}")
    df_bronze.display()